# TickBuffer smoke benchmark: pure Python vs pybind11

Minimal pure-Python `TickBuffer` (deque of tuples, core methods only) checked against the C++ `feature.TickBuffer`, then compared on wall time and memory.

In [1]:
import gc
import math
import statistics
import time
import tracemalloc
from collections import deque
from itertools import islice

import numpy as np
import psutil

from feature import TickBuffer

## 1. Pure-Python TickBuffer

In [ ]:
class PyTickBuffer:
    def __init__(self, capacity):
        self.ticks = deque(
            maxlen=capacity
        )  # (mid, spread, imbalance, timestamp, volume)

    def push(self, mid, spread, imbalance, timestamp, volume=0.0):
        self.ticks.append((mid, spread, imbalance, timestamp, volume))

    def __len__(self):
        return len(self.ticks)

    def vwap(self, n):
        last = list(islice(self.ticks, max(0, len(self.ticks) - n), None))
        v = sum(t[4] for t in last)
        return sum(t[0] * t[4] for t in last) / v if v > 0 else math.nan

    def mid_zscore(self, window):
        if len(self.ticks) < window:
            return math.nan
        mids = [t[0] for t in islice(self.ticks, len(self.ticks) - window, None)]
        sd = statistics.stdev(mids)
        return (mids[-1] - statistics.fmean(mids)) / sd if sd > 1e-9 else 0.0

    def imbalance_ewma(self, alpha):
        it = iter(self.ticks)
        e = next(it, (math.nan,) * 5)[
            2
        ]  # (math.nan,) * 5 in case self.ticks is completely empty
        for t in it:
            e += alpha * (t[2] - e)
        return e

## 2. Synthetic ticks and correctness check

In [3]:
N = 1_000_000  # ticks pushed
CAPACITY = 100_000  # ring size, so the buffer wraps 10x
WINDOW, ALPHA = 1_000, 0.05

rng = np.random.default_rng(0)
mid = 0.5 + 0.1 * np.tanh(
    np.cumsum(rng.normal(0, 1e-3, N))
)  # bounded random walk in (0.4, 0.6)
spread = rng.uniform(0.001, 0.02, N)
imbalance = rng.uniform(-1, 1, N)
timestamp = 1_790_000_000_000 + np.arange(N, dtype=np.int64) * 100
volume = rng.exponential(50, N)
cols = [
    a.tolist() for a in (mid, spread, imbalance, timestamp, volume)
]  # plain Python scalars

py_buf, cpp_buf = PyTickBuffer(CAPACITY), TickBuffer(CAPACITY)
for row in zip(*cols):
    py_buf.push(*row)
cpp_buf.extend(mid, spread, imbalance, timestamp, volume)

for name, args in [
    ("vwap", (WINDOW,)),
    ("mid_zscore", (WINDOW,)),
    ("imbalance_ewma", (ALPHA,)),
]:
    a, b = getattr(py_buf, name)(*args), getattr(cpp_buf, name)(*args)
    assert math.isclose(a, b, rel_tol=1e-9), (name, a, b)
    print(f"{name:15s} py={a:.10f}  cpp={b:.10f}")
assert len(py_buf) == len(cpp_buf) == CAPACITY

vwap            py=0.5748631750  cpp=0.5748631750
mid_zscore      py=1.7744578722  cpp=1.7744578722
imbalance_ewma  py=0.0650872188  cpp=0.0650872188


## 3. Time

In [6]:
def bench(fn, repeat=5):
    """Best-of-`repeat` wall time in seconds."""
    best = math.inf
    for _ in range(repeat):
        t0 = time.perf_counter()
        fn()
        best = min(best, time.perf_counter() - t0)
    return best


def push_all(buf):
    push = buf.push
    for row in zip(*cols):
        push(*row)


cases = {
    f"push x{N:,}": (
        lambda: push_all(PyTickBuffer(CAPACITY)),
        lambda: push_all(TickBuffer(CAPACITY)),
    ),
    f"vwap({WINDOW})": (lambda: py_buf.vwap(WINDOW), lambda: cpp_buf.vwap(WINDOW)),
    f"mid_zscore({WINDOW})": (
        lambda: py_buf.mid_zscore(WINDOW),
        lambda: cpp_buf.mid_zscore(WINDOW),
    ),
    f"imbalance_ewma (n={CAPACITY:,})": (
        lambda: py_buf.imbalance_ewma(ALPHA),
        lambda: cpp_buf.imbalance_ewma(ALPHA),
    ),
}

print(f"{'case':32s} {'python':>12s} {'pybind':>12s} {'speedup':>9s}")
for name, (py_fn, cpp_fn) in cases.items():
    tp, tc = bench(py_fn, 3 if name.startswith("push") else 20), bench(
        cpp_fn, 3 if name.startswith("push") else 20
    )
    print(f"{name:32s} {tp * 1e3:10.3f}ms {tc * 1e3:10.3f}ms {tp / tc:8.1f}x")

# Bulk numpy ingest only exists on the C++ side
t_ext = bench(
    lambda: TickBuffer(CAPACITY).extend(mid, spread, imbalance, timestamp, volume)
)
print(f"{'pybind extend() from numpy':32s} {'-':>12s} {t_ext * 1e3:10.3f}ms")

case                                   python       pybind   speedup
push x1,000,000                     447.330ms    626.673ms      0.7x
vwap(1000)                            2.449ms      0.019ms    130.3x
mid_zscore(1000)                      4.390ms      0.026ms    169.5x
imbalance_ewma (n=100,000)           12.875ms      1.134ms     11.4x
pybind extend() from numpy                  -     50.889ms


## 4. Storage

`tracemalloc` only sees Python-heap allocations, so the C++ buffer is measured by process RSS delta instead (also shown for Python). The C++ lower bound is 5 x 8 bytes = 40 B/tick.

In [5]:
MEM_CAP = 1_000_000
proc = psutil.Process()


def measure(make):
    gc.collect()
    rss0 = proc.memory_info().rss
    tracemalloc.start()
    buf = make()
    traced, _ = tracemalloc.get_traced_memory()
    tracemalloc.stop()
    gc.collect()
    return buf, traced, proc.memory_info().rss - rss0


def make_py():
    b = PyTickBuffer(MEM_CAP)
    # rebuild scalars inside the measured region so the buffer owns its float/int objects
    for row in zip(*(a.tolist() for a in (mid, spread, imbalance, timestamp, volume))):
        b.push(*row)
    return b


def make_cpp():
    b = TickBuffer(MEM_CAP)
    b.extend(mid, spread, imbalance, timestamp, volume)
    return b


print(f"{'impl':8s} {'tracemalloc':>14s} {'RSS delta':>12s} {'B/tick (RSS)':>13s}")
for name, make in [("python", make_py), ("pybind", make_cpp)]:
    buf, traced, rss = measure(make)
    assert len(buf) == MEM_CAP
    print(
        f"{name:8s} {traced / 2**20:11.1f}MiB {rss / 2**20:9.1f}MiB {rss / MEM_CAP:12.1f}"
    )
    del buf

impl        tracemalloc    RSS delta  B/tick (RSS)
python         206.2MiB     237.3MiB        248.8
pybind           0.0MiB      38.2MiB         40.0
